# kCN wavelength-dependent pump-probe analysis — 100 K

Analysis of the October 26, 2023 kCN MIR-idler dataset as a function of pump frequency.  
The notebook loads the transient reflectivity data, fits the non-oscillatory response,
extracts coherent phonon residuals, and compares the fitted quantities with the
equilibrium optical conductivity.


In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import lmfit
import os



## 1. Setup

Define the local data root, load the project analysis packages, and set shared plotting defaults.


In [ ]:
import Idler_package as p
import osc_fit_package as op


In [ ]:
# Default plot parameters
mpl.rcParams['xtick.labelsize'] = 10
mpl.rcParams['ytick.labelsize'] = 10
mpl.rcParams['axes.labelsize'] = 12
mpl.rcParams['figure.titlesize'] = 14
mpl.rcParams['axes.titlesize'] = 12
mpl.rcParams['lines.linewidth'] = 2


In [ ]:
root_eq = os.getcwd() + '/' + 'equilibrium_data'
root_pp = os.getcwd() + '/' + 'data'


## 2. Equilibrium reference data

Load the 100 K, b-axis equilibrium optical conductivity used as the spectral reference
for the pump-frequency-dependent analysis.


In [ ]:
temp = [297, 200, 100, 50, 30, 12.6]

T_i = 2
ax = 'b'

data_eq_CN = np.loadtxt(
    root_eq
    + '/kCN_'
    + str(temp[T_i]) + 'K_' + ax + '_eq.dat'
)

w_CN = data_eq_CN[:, 0]
s1_CN = data_eq_CN[:, 1]


## 3. Pump-probe dataset

Measurement metadata for the October 26, 2023 frequency-dependent scan. `I0` contains
the normalization values associated with each pump-frequency measurement.


In [ ]:
date = [20231026, 20231026, 20231026, 20231026, 20231026, 20231026, 20231026]
time = [132006, 142058, 165322, 181910, 193205, 203800, 214254]
n_cycles = [10, 10, 10, 10, 7, 10, 10]
n_angles = [1, 1, 1, 1, 1, 1, 1]
freq_0 = [44.5, 33, 41.5, 36, 38.5, 40, 39]

I0 = np.array([8.88, 8.3, 7.9, 8.16, 8.4, 8.32, 8.1]) * 1e-4

key = p.keys(freq_0, 'THz')
var = 'Freq. [THz]'
sample = 'kCN'


### Analysis switches

These flags control optional analysis and plotting sections later in the notebook.


In [ ]:
generate_fit_comps = False
coherent_phonon_analysis = True
phonon_maps = True
param_extract = True
log_plot = False


### Compile the transient reflectivity data

`p.compile_data` returns the mean transient, pump-probe delay axis, normalized transient,
and the pump-frequency array.


In [ ]:
PP_Mean, PP_delay, PP_Norm, freq = p.compile_data(
    root_pp, date, time, n_cycles, n_angles, I0, freq_0, key, sample
)


### Uncertainty from repeated scans

For each condition, `p.compile_std` calculates the **standard deviation across scans** (`ddof=0`, as defined in the Idler package). I convert this to the **standard error of the mean** (SEM) using the actual number of scans and cycles. The baseline is the fifth point from the end of the transient; an approximate baseline-corrected SEM is obtained by adding the two pointwise variances in quadrature. Correlation between them is not available from the compiled standard deviations, so these are approximate error bars. If there is only one scan, SEM is left undefined.

The error bars on normalized traces divide the baseline-corrected SEM by the absolute minimum of the mean transient; they **do not** include uncertainty in that fitted-from-data normalization factor. Fit-parameter error bars later in the notebook come from `lmfit` covariance estimates, not these scan SEMs.

In [ ]:
# Scan-to-scan standard errors (SEM); no changes to PP_Mean, PP_Norm, or freq.
import pandas as pd

PP_std = p.compile_std(root_pp, date, time, n_cycles, n_angles, I0, freq_0, key, sample)

# Match the dataset ordering used inside p.compile_data / p.compile_std.
_sort_idx = np.argsort(np.asarray(freq_0))
_sem_date = np.asarray(date)[_sort_idx]
_sem_time = np.asarray(time)[_sort_idx]
_sem_cycles = np.asarray(n_cycles)[_sort_idx]

PP_sem = {}             # SEM of the averaged, baseline-corrected signal
PP_Norm_sem = {}        # Approximate SEM after normalization
peak_dr_r_sem = []      # SEM at the minimum of each mean trace
pos_time_sem = []       # SEM at the fixed positive-time point (index 125)

for i, k in enumerate(key):
    # The number of scans is encoded in column 1 of the first CSV file.
    _csv = (root_pp + str(_sem_date[i]) + '/' + sample + '_MIR-Idler_PP'
            + str(_sem_date[i]) + '_' + str(_sem_time[i]) + '_angle0_scan0.csv')
    _header = pd.read_csv(_csv, skiprows=1, header=0, usecols=[1])
    _n_scans = int(np.max(_header.iloc[:, 0].to_numpy())) + 1
    _n_total = _n_scans * int(_sem_cycles[i])

    # PP_std is the across-scan standard deviation, not the SEM.
    _sem_raw = np.asarray(PP_std[k]) / np.sqrt(_n_total) if _n_total > 1 else np.full_like(PP_std[k], np.nan)
    _sem_baseline_corrected = np.sqrt(_sem_raw**2 + _sem_raw[-5]**2)
    PP_sem[k] = _sem_baseline_corrected
    PP_Norm_sem[k] = _sem_baseline_corrected / abs(PP_Mean[k].min())
    peak_dr_r_sem.append(_sem_baseline_corrected[np.argmin(PP_Mean[k])])
    pos_time_sem.append(PP_Norm_sem[k][125])

peak_dr_r_sem = np.asarray(peak_dr_r_sem)
pos_time_sem = np.asarray(pos_time_sem)
print('Peak reflectivity SEM:', peak_dr_r_sem)

### Plot colors

Define a discrete color set for the pump-frequency series.


In [ ]:
c1 = 'slateblue'
c2 = 'red'

colors = p.colorFader(len(freq), c1, c2)
colors1 = plt.cm.rainbow(np.linspace(0, 1, len(freq)))


## 4. Inspect the transient response

Plot the raw mean transients, peak response, equilibrium comparison, and normalized traces
before fitting.


In [ ]:
plt.figure(figsize=(5, 10))
plt.title('Mean', fontsize=14)
plt.xlabel('PP_delay [ps]')
plt.ylabel('$\Delta$R/R')

for i in range(len(freq)):
    plt.fill_between(
        PP_delay[key[i]],
        (PP_Mean[key[i]] - PP_sem[key[i]]) * 100 + i * 5,
        (PP_Mean[key[i]] + PP_sem[key[i]]) * 100 + i * 5,
        color=colors1[i], alpha=0.15, linewidth=0
    )
    plt.plot(
        PP_delay[key[i]],
        PP_Mean[key[i]] * 100 + i * 5,
        lw=1,
        label=key[i],
        color=colors1[i]
    )

plt.legend(fontsize=9, loc='lower right')
plt.show()


In [ ]:
plt.figure(figsize = (5,4))
# plt.title(" (Peak reflectivity)",fontsize=14)
plt.xlabel(var)
plt.ylabel('Peak $\Delta$R/R')
for i in range(len(freq)):
    plt.errorbar(freq[i], PP_Mean[key[i]].min(), yerr=peak_dr_r_sem[i],
                 fmt='o', markersize=5, capsize=3, lw=1, color=colors[i])
    print(freq[i],key[i])
plt.show()

peak_dr_r = []
for i in range(len(freq)):
    peak_dr_r.append(PP_Mean[key[i]].min())
print(peak_dr_r)

In [ ]:
fig, ax1 = plt.subplots(figsize=(6, 3))

ax1.set_ylabel('$\sigma_1$ $[\Omega*cm]^{-1}$', color='grey')
ax1.set_xlabel('Pump energy [eV]', color='k')
ax1.plot(w_CN * 0.00414, s1_CN, label='100K, b-axis', lw=2.5, color='darkgray', alpha=0.6)
ax1.tick_params(axis='y', labelcolor='k')
ax1.tick_params(direction='in', axis='both', labelcolor='k')
ax1.set_xlim(0.12, 0.2)

ax2 = ax1.twinx()
ax2.set_xlabel('Pump energy [eV]')
ax2.set_ylabel('Peak $\Delta$R/R', color='crimson')
ax2.tick_params(direction='in', axis='both', labelcolor='k')
ax2.errorbar(np.array(freq) * 0.00414, peak_dr_r, yerr=peak_dr_r_sem,
             fmt='o', markersize=5, capsize=3, color='crimson', lw=1.5)
ax2.tick_params(axis='y', labelcolor='k')

fig.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(5, 4))
plt.title('$\kappa$-(BEDT-TTF)$_2$Cu$_2$(CN)$_3$', fontsize=14)
plt.xlim(-1, 5)

for i in range(len(freq)):
    plt.fill_between(
        PP_delay[key[i]],
        PP_Norm[key[i]] - PP_Norm_sem[key[i]],
        PP_Norm[key[i]] + PP_Norm_sem[key[i]],
        color=colors1[i], alpha=0.15, linewidth=0
    )
    plt.plot(
        PP_delay[key[i]],
        PP_Norm[key[i]],
        lw=0.9,
        label=str(freq[i]) + ' THz',
        color=colors1[i]
    )

plt.vlines(PP_delay[key[i]][125], -0.6, -0.2, lw=1.5, color='purple')
plt.legend(fontsize=10, loc='lower right')
plt.xlabel('PP_delay [ps]')
plt.ylabel('Normalized ΔR/R')
plt.show()

plt.figure(figsize=(4, 2))
plt.title('$\kappa$-(BEDT-TTF)$_2$Cu$_2$(CN)$_3$', fontsize=14)
plt.xlabel(var)
plt.ylabel('Pos. time\nΔR/R')

for i in range(len(freq)):
    plt.errorbar(freq[i], PP_Norm[key[i]][125], yerr=pos_time_sem[i],
                 fmt='o', markersize=5, capsize=3, lw=1, color='purple')

plt.show()


In [ ]:
print(freq)
print(key)

## 5. Fit the non-oscillatory transient

Fit each transient using the relaxation model defined in `Idler_package`. The residual
between the data and the non-oscillatory fit is retained as the coherent-phonon signal.

The FFT is evaluated over sample indices `20:140`, matching the existing analysis.


In [ ]:
params = lmfit.Parameters()
params.add('b', min=-1, max=1, value=0)
params.add('m', vary=False, value=0)
params.add('td', min=-1, max=1, value=-0.1)
params.add('tau_0', min=0, max=10, value=0.2)
params.add('tau_1', min=0, max=10, value=0.2)
params.add('tau_2', min=0, max=10, value=1.2)
params.add('A0', min=-1, max=0, value=-0.14)
params.add('A1', min=-2, max=0, value=-1.4)
params.add('A2', min=-1, max=0, value=-0.4)

freq_except = []

pretty = False
guess_pl = True
fit_pl = True
osc_pl = False
FT_pl = False

fit_par, THz_ax, phonon_mag, phonon_phase, t, phonon = p.fit_compiler(
    20, 140, params, PP_Mean, PP_delay, freq, key, freq_except,
    pretty, guess_pl, fit_pl, osc_pl, FT_pl
)


In [ ]:
print(fit_par.keys())

### Optional fit-component export

When `generate_fit_comps` is enabled, inspect the fitted and initial-guess model components
for a selected pump frequency.


In [ ]:
if generate_fit_comps:
    print(key)

    n = 0
    Fit_Comps = p.fit_comps(
        params,
        PP_Norm[key[n]],
        PP_delay[key[n]],
        key[n],
        guess_comps=False,
        fit_comps=True
    )
    print(Fit_Comps)

    Guess_Comps = p.fit_comps(
        params,
        PP_Norm[key[n]],
        PP_delay[key[n]],
        key[n],
        guess_comps=True,
        fit_comps=False
    )
    print(Guess_Comps)


**Uncertainty note:** The Fourier magnitudes, phases, residual-peak amplitudes, and FFT-index intensities below are derived from a single averaged trace per pump condition. The notebook does not have scan-resolved FFT estimates or a propagated fit covariance for these observables, so no statistically justified error bars are added to those figures.

## 6. Coherent phonon analysis

Inspect the Fourier magnitude, relative phase, pump-frequency dependence, and time-domain
fit residuals. Enable this section with `coherent_phonon_analysis` in the analysis switches.


In [ ]:
if coherent_phonon_analysis:
    print(freq)

    plt.figure(figsize=(6, 3))
    plt.tick_params(axis='both', direction='in', length=6, color='black')
    plt.xlabel('Phonon freq. [THz]')
    plt.ylabel('$I_{FT}$ [arb. units]')
    plt.ylim(0, 0.08)
    plt.xlim(0, 2.7)

    for i in range(len(freq)):
        plt.plot(
            THz_ax[key[i]][0:40],
            phonon_mag[key[i]][0:40],
            label=str(key[i]),
            lw=1,
            c=colors1[i]
        )
        plt.fill_between(
            x=THz_ax[key[i]][0:40],
            y1=phonon_mag[key[i]][0:40],
            y2=0,
            color=colors1[i],
            alpha=0.2
        )

    plt.vlines(1.75, -0.02, 0.05, lw=2, color='purple')
    plt.legend()
    plt.show()

    plt.figure(figsize=(5, 4))
    plt.xlabel('Phonon freq. [THz]')
    plt.ylabel('FT phase [arb. units]')

    for i in range(len(key)):
        relative_phase = np.unwrap(
            phonon_phase[key[i]][0:40] - phonon_phase[key[-1]][0:40]
        )
        plt.plot(
            THz_ax[key[i]][0:40],
            relative_phase,
            label=str(key[i]),
            lw=1,
            c=colors1[i]
        )

    plt.legend()
    plt.show()

    plt.figure(figsize=(5, 3))
    plt.xlabel('Pump freq. [THz]')
    plt.ylabel('Peak $I_{FT}$ [arb. units]')

    for i in range(len(freq)):
        plt.scatter(
            freq[i],
            phonon_mag[key[i]][0:40].max(),
            lw=1,
            color='purple'
        )

    plt.show()

    plt.figure(figsize=(6, 3))
    plt.xlabel('PP delay [ps]')
    plt.ylabel('arb. units')
    plt.xlim(-3, 6)

    for i in range(len(key)):
        plt.plot(
            t[key[i]][20:140],
            phonon[key[i]][20:140] * 100,
            label=str(key[i]),
            lw=2,
            color=colors1[i]
        )

    plt.legend()
    plt.show()


### Residual oscillation amplitude

Estimate the maximum absolute residual amplitude over the selected positive-time window.


In [ ]:
osc_am = []

for i in range(len(freq)):
    osc_am.append(np.abs(phonon[key[i]][20:130]).max())

print(osc_am)
plt.plot(freq, osc_am)
plt.show()


### Phonon amplitude and phase maps

Construct two-dimensional maps of Fourier magnitude and relative phase versus pump frequency.


In [ ]:
if phonon_maps:
    eV, eV_ax, FT_map, phase_map = p.phonon_maps(
        phonon_mag, phonon_phase, THz_ax, freq, key
    )

    print(FT_map.shape)

    plt.figure()
    plt.title('Phonon mag. map')

    im1 = plt.imshow(
        FT_map[0:23, :],
        aspect='auto',
        cmap=plt.get_cmap('PRGn_r'),
        extent=[
            eV[0],
            eV[-1],
            THz_ax[key[-1]][23],
            THz_ax[key[-1]][0]
        ],
        vmin=-np.max(np.abs(FT_map)),
        vmax=np.max(np.abs(FT_map))
    )

    plt.colorbar(im1, shrink=1, aspect=20)
    plt.xlabel('Pump energy [eV]')
    plt.ylabel('Phonon freq. [THz]')
    plt.tight_layout()
    plt.show()

    print(phase_map.shape)

    plt.figure()
    plt.title('Phonon phase map')

    im1 = plt.imshow(
        phase_map[0:25, :],
        aspect='auto',
        cmap=plt.cm.seismic,
        extent=[
            freq[0],
            freq[-1],
            THz_ax[key[-1]][25],
            THz_ax[key[-1]][0]
        ],
        vmin=-np.max(np.abs(phase_map[0:30])),
        vmax=np.max(np.abs(phase_map[0:30]))
    )

    plt.colorbar(im1, shrink=1, aspect=20)
    plt.xlabel('Pump freq. [THz]')
    plt.ylabel('Phonon freq. [THz]')
    plt.tight_layout()
    plt.show()


## 7. Extract relaxation-fit parameters

Extract the varying lmfit parameters and uncertainties, then inspect their pump-frequency
dependence.


In [ ]:
if param_extract:
    parameter_lists, stderr_lists = p.extract_fit_params(fit_par, key)
    print(freq, fit_par.keys(), fit_par[key[0]].keys())

    plt.rcParams['ytick.right'] = plt.rcParams['ytick.labelright'] = True
    plt.rcParams['ytick.left'] = plt.rcParams['ytick.labelleft'] = False

    for parameter in fit_par[key[0]].keys():
        if fit_par[key[0]][parameter].vary:
            plt.figure(figsize=(4, 3))
            plt.ylabel(str(parameter))
            plt.xlabel(var)
            plt.errorbar(
                freq,
                parameter_lists[parameter],
                xerr=1,
                yerr=stderr_lists[parameter],
                marker='o',
                markersize=6,
                lw=1,
                linestyle='none',
                color='purple'
            )
            plt.show()

    tau0 = np.mean(parameter_lists['tau_0'])
    print(tau0)

    plt.figure(figsize=(5, 2))
    plt.tick_params(axis='both', direction='in', length=6, color='black')
    plt.ylabel('$\\tau_{fast}$ [ps]')
    plt.xlabel('Pump energy [eV]')
    plt.errorbar(
        freq * 0.00414,
        np.array(parameter_lists['tau_1']),
        xerr=0.004,
        yerr=stderr_lists['tau_1'],
        marker='o',
        markersize=8,
        lw=1.5,
        linestyle='none',
        color='crimson'
    )
    plt.plot(freq * 0.00414, np.array(parameter_lists['tau_1']), color='crimson', lw=2)
    plt.vlines(0.1615, 0.18, 0.26, color='green', linestyle='dashed', lw=2)
    plt.show()

    plt.figure(figsize=(5, 2))
    plt.tick_params(axis='both', direction='in', length=6, color='black')
    plt.ylabel('$(\Delta R/R)_{fast}$')
    plt.xlabel('Pump energy [eV]')
    plt.errorbar(
        freq * 0.00414,
        np.array(parameter_lists['A1']) * 0.5,
        xerr=0.004,
        yerr=np.array([np.nan if e is None else abs(e) * 0.5
                       for e in stderr_lists['A1']]),
        marker='o',
        markersize=8,
        lw=1.5,
        linestyle='none',
        color='crimson'
    )
    plt.plot(freq * 0.00414, np.array(parameter_lists['A1']) * 0.5, color='crimson', lw=2)
    plt.vlines(0.1615, -0.064, -0.052, color='green', linestyle='dashed', lw=2)
    plt.show()


## 8. Compare pump-probe observables with equilibrium conductivity

Convert pump frequency to photon energy and compare the transient-fit observables with the
100 K equilibrium conductivity spectrum.


In [ ]:
peak_dr_r = [PP_Mean[key[i]].min() * -1 for i in range(len(freq))]
print(peak_dr_r)

eV_eq_CN = w_CN * 0.00414
eV_pp = freq * 0.00414


In [ ]:
fig, ax1 = plt.subplots(figsize=(5, 3))
ax1.set_title('kCN,RT')
ax1.set_ylabel('$\sigma_1$ $[\Omega*cm]^{-1}$', color='grey')
ax1.set_xlabel('Pump energy [eV]', color='k')
ax1.plot(eV_eq_CN, s1_CN, label='100K, b-axis', lw=2.5, color='darkgray', alpha=0.6)
ax1.tick_params(axis='y', labelcolor='k')
ax1.set_xscale('log')
ax1.set_xlim(0.09, 0.35)
ax1.set_ylim(-50, 800)

xtick_locations = [0.1, 0.16, 0.2, 0.3]
ytick_locations = [0.2, 0.3]

ax2 = ax1.twinx()
ax2.set_xlabel('Pump energy [eV]')
ax2.set_ylabel('$\\tau_{fast}$ [ps]', color='crimson')
ax2.errorbar(
    freq * 0.00414,
    np.array(parameter_lists['tau_1']),
    xerr=0.004,
    yerr=stderr_lists['tau_1'],
    marker='o',
    markersize=8,
    lw=1.5,
    linestyle='none',
    color='crimson'
)
ax2.tick_params(axis='y', labelcolor='k')
ax2.set_ylim(0.15, 0.32)

ax2.set_xticks(xtick_locations)
ax2.set_xticklabels([str(tick) for tick in xtick_locations])
ax2.set_yticks(ytick_locations)
ax2.set_yticklabels([str(tick) for tick in ytick_locations])

ax1.tick_params(direction='in', axis='both', labelcolor='k')
ax2.tick_params(direction='in', axis='both', labelcolor='k')

fig.tight_layout()
plt.show()


In [ ]:
fig, ax1 = plt.subplots(figsize=(5, 3))
ax1.set_title('kCN,RT')
ax1.set_ylabel('$\sigma_1$ $[\Omega*cm]^{-1}$', color='grey')
ax1.set_xlabel('Pump energy [eV]', color='k')
ax1.plot(eV_eq_CN, s1_CN, label='100K, b-axis', lw=2.5, color='darkgray', alpha=0.6)
ax1.tick_params(axis='y', labelcolor='k')
ax1.set_xscale('log')
ax1.set_xlim(0.09, 0.35)
ax1.set_ylim(-50, 800)

xtick_locations = [0.1, 0.16, 0.2, 0.3]
ytick_locations = [1.5,1.75]

ax2 = ax1.twinx()
ax2.set_xlabel('Pump energy [eV]')
ax2.set_ylabel('$\\tau_{slow}$ [ps]', color='crimson')
ax2.errorbar(
    freq * 0.00414,
    np.array(parameter_lists['tau_2']),
    xerr=0.004,
    yerr=stderr_lists['tau_2'],
    marker='o',
    markersize=8,
    lw=1.5,
    linestyle='none',
    color='crimson'
)
ax2.tick_params(axis='y', labelcolor='k')
ax2.set_ylim(1.4, 1.9)

ax2.set_xticks(xtick_locations)
ax2.set_xticklabels([str(tick) for tick in xtick_locations])
ax2.set_yticks(ytick_locations)
ax2.set_yticklabels([str(tick) for tick in ytick_locations])

ax1.tick_params(direction='in', axis='both', labelcolor='k')
ax2.tick_params(direction='in', axis='both', labelcolor='k')

fig.tight_layout()
plt.show()


In [ ]:
fig, ax1 = plt.subplots(figsize=(5, 3))
ax1.set_title('kCN,RT')
ax1.set_ylabel('$\sigma_1$ $[\Omega*cm]^{-1}$', color='grey')
ax1.set_xlabel('Pump energy [eV]', color='k')
ax1.plot(eV_eq_CN, s1_CN, label='100K, b-axis', lw=2.5, color='darkgray', alpha=0.6)
ax1.tick_params(axis='y', labelcolor='k')
ax1.set_xscale('log')
ax1.set_xlim(0.09, 0.35)
ax1.set_ylim(-50, 800)

xtick_locations = [0.1, 0.16, 0.2, 0.3]
ytick_locations = [0.06, 0.08, 0.1]

ax2 = ax1.twinx()
ax2.set_xlabel('Pump energy [eV]')
ax2.set_ylabel('($-\Delta$R/R)$_{peak}$', color='crimson')
ax2.errorbar(eV_pp, peak_dr_r, yerr=peak_dr_r_sem, fmt='o',
             markersize=5, capsize=3, color='crimson', lw=1.5)
ax2.tick_params(axis='y', labelcolor='k')
ax2.set_ylim(0.06, 0.1)

ax2.set_xticks(xtick_locations)
ax2.set_xticklabels([str(tick) for tick in xtick_locations])
ax2.set_yticks(ytick_locations)
ax2.set_yticklabels([str(tick) for tick in ytick_locations])

ax1.tick_params(direction='in', axis='both', labelcolor='k')
ax2.tick_params(direction='in', axis='both', labelcolor='k')

fig.tight_layout()
plt.show()


##### 9. Fourier intensity near the selected phonon mode

Inspect the squared Fourier magnitude and extract the intensity at FFT index 7, corresponding
to the selected coherent mode in the existing analysis.


In [ ]:
plt.figure(figsize=(6, 3))
plt.tick_params(axis='both', direction='in', length=6, color='black')
plt.xlabel('Phonon freq. [THz]')
plt.ylabel('$I_{FT}$ [arb. units]')
plt.ylim(0, 0.002)
plt.xlim(0, 2.7)

for i in range(len(freq)):
    plt.plot(
        THz_ax[key[i]][0:40],
        phonon_mag[key[i]][0:40] ** 2,
        label=str(key[i]),
        lw=1,
        c=colors1[i]
    )
    plt.fill_between(
        x=THz_ax[key[i]][0:40],
        y1=phonon_mag[key[i]][0:40] ** 2,
        y2=0,
        color=colors1[i],
        alpha=0.2
    )

plt.vlines(THz_ax[key[0]][7], -0.02, 0.5, lw=2, color='purple')
print(THz_ax[key[0]][7])
plt.legend()
plt.show()

plt.figure(figsize=(5, 3))
plt.xlabel('Pump freq. [THz]')
plt.ylabel('Peak $I_{FT}$ [arb. units]')

for i in range(len(freq)):
    plt.scatter(freq[i], phonon_mag[key[i]][7] ** 2, lw=1, color='purple')

plt.show()


In [ ]:
Ift = []

for i in range(len(freq)):
    Ift.append(
        phonon_mag[key[i]][7]
        * (-t[key[0]][1] + t[key[0]][0])
        / ((t[key[0]][20] - t[key[0]][140]) / 2)
    )

print(Ift)
plt.plot(freq, Ift)
plt.show()


In [ ]:
fig, ax1 = plt.subplots(figsize=(5, 3))
ax1.set_title('kCN,RT')
ax1.set_ylabel('$\sigma_1$ $[\Omega*cm]^{-1}$', color='grey')
ax1.set_xlabel('Pump energy [eV]', color='k')
ax1.plot(eV_eq_CN, s1_CN, label='100K, b-axis', lw=2.5, color='darkgray', alpha=0.6)
ax1.tick_params(axis='y', labelcolor='k')
ax1.set_xscale('log')
ax1.set_xlim(0.09, 0.35)
ax1.set_ylim(-50, 800)

xtick_locations = [0.1, 0.16, 0.2, 0.3]
ytick_locations = [0.0002, 0.0006, 0.001]

ax2 = ax1.twinx()
ax2.set_xlabel('Pump energy [eV]')
ax2.set_ylabel('$I_{1.1THz}$ [a.u.]', color='crimson')
ax2.scatter(eV_pp, Ift, color='crimson', lw=2)
ax2.tick_params(axis='y', labelcolor='k')
ax2.set_ylim(0.0001, 0.0012)

ax2.set_xticks(xtick_locations)
ax2.set_xticklabels([str(tick) for tick in xtick_locations])
ax2.set_yticks(ytick_locations)
ax2.set_yticklabels([str(tick) for tick in ytick_locations])

ax1.tick_params(direction='in', axis='both', labelcolor='k')
ax2.tick_params(direction='in', axis='both', labelcolor='k')

fig.tight_layout()
plt.show()


## 10. Optional logarithmic transient plot

Plot the normalized transients on a logarithmic scale after shifting by the fitted time-zero
parameter. Enable with `log_plot` in the analysis switches.


In [ ]:
if log_plot:
    plt.figure(figsize=(4, 6))
    plt.title('$\kappa$-(BEDT-TTF)$_2$Cu$_2$(CN)$_3$', fontsize=14)
    plt.xlim(-1, 3)
    plt.ylim(1.2e-1, 1.05)
    plt.yscale('log', base=10)
    plt.tick_params(axis='both', direction='in', length=6, color='black')

    for i in range(len(freq)):
        delay_shift = 0

        if freq[i] == 33:
            delay_shift = 0.03
        elif freq[i] == 36:
            delay_shift = 0.02

        plt.plot(
            PP_delay[key[i]] - parameter_lists['td'][i] + delay_shift,
            PP_Norm[key[i]],
            lw=0.9,
            label=str(freq[i]) + ' THz',
            color=colors1[i]
        )

    plt.vlines(PP_delay[key[i]][125], 2.5e-1, 0.5, lw=1.5, color='purple')
    plt.legend(fontsize=9, loc='upper right')
    plt.xlabel('PP_delay [ps]')
    plt.ylabel('Normalized -ΔR/R')
    plt.show()


## 11. Explicit coherent-phonon fitting

Fit the residual oscillations with two damped phonon modes using `osc_fit_package.fit_osc`.
The initial amplitudes, phases, lifetimes, rise times, and frequencies are defined below.


In [ ]:
num_ph = 2

params1 = lmfit.Parameters()
params1.add('a1', min=0, max=0.02, value=0.006982)
params1.add('a2', min=0, max=0.05, value=0.002489)

params1.add('phi1', min=-100, max=100, value=-0.9428)
params1.add('phi2', min=-100, max=100, value=-50.43)

params1.add('tau1', min=0, max=10, value=2.644)
params1.add('tau2', min=0, max=3, value=0.4534)

params1.add('tau_r1', min=0, max=5, value=5)
params1.add('tau_r2', min=0, max=5, value=0.1226)

params1.add('w1', min=1, max=2.5, value=1.13)
params1.add('w2', min=1.5, max=2.5, value=1.846)

fit_par_osc = {}

for i in range(len(key)):
    fit_par_osc[key[i]] = op.fit_osc(
        5,
        140,
        params1,
        phonon[key[i]],
        t[key[i]],
        num_ph,
        None,
        True,
        False,
        True
    )


### Oscillation-fit parameters versus pump frequency

Extract the fitted phonon parameters and plot each varying parameter as a function of pump
frequency.


In [ ]:
parameter_lists_osc, stderr_lists_osc = p.extract_fit_params(fit_par_osc, key)

print(freq, fit_par_osc.keys(), fit_par_osc[key[0]].keys())
print(list(fit_par_osc.keys()))

plt.rcParams['ytick.right'] = plt.rcParams['ytick.labelright'] = True
plt.rcParams['ytick.left'] = plt.rcParams['ytick.labelleft'] = False

for parameter in fit_par_osc[key[0]].keys():
    if fit_par_osc[key[0]][parameter].vary:
        plt.figure(figsize=(4, 3))
        plt.ylabel(str(parameter))
        plt.xlabel(var)
        # lmfit may return None when covariance cannot be estimated.
        _fit_err = np.array([np.nan if e is None else e
                             for e in stderr_lists_osc[parameter]], dtype=float)
        plt.errorbar(freq, parameter_lists_osc[parameter], yerr=_fit_err,
                     fmt='none', ecolor='gray', elinewidth=1, capsize=3, zorder=1)
        plt.scatter(freq, parameter_lists_osc[parameter],
                    color=colors[:len(freq)], zorder=2)
        plt.show()
